In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.dataset import ImageDataset
from torch.utils.data import DataLoader

annotations_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annotations_file_trainval, img_dir_trainval)
trainval_dl = DataLoader(trainval_dataset, batch_size=32, shuffle=True)

In [3]:
from src.model import Model

model = Model()

In [4]:
from src.configs import S, B, C, IDX_TO_CLASS
from src.utils import convert_xywh_coords

def decode_preds(preds_batch):
    decoded_preds = []

    for pred in preds_batch:
        pred = pred.reshape((S, S, C + B * 5))
        objects = []

        for i in range(S):
            for j in range(S):
                pred_cell = pred[i][j]

                pred_class_idx = pred_cell[:20].argmax().item()
                pred_class_prob = pred_cell[pred_class_idx].item()
                pred_class = IDX_TO_CLASS[pred_class_idx]
                
                pred_1_confidence = (pred_cell[24].item() * \
                                     pred_class_prob,)
                pred_2_confidence = (pred_cell[29].item() * \
                                     pred_class_prob,)

                bbox_1 = convert_xywh_coords(pred_cell[20:24], i, j, False, True)
                bbox_2 = convert_xywh_coords(pred_cell[25:29], i, j, False, True)
                
                objects.append((pred_class,) + pred_1_confidence + bbox_1)
                objects.append((pred_class,) + pred_2_confidence + bbox_2)

        decoded_preds.append(objects)

    return decoded_preds

In [5]:
X_batch, y_batch = next(iter(trainval_dl))

X_batch.shape, y_batch.shape

(torch.Size([32, 3, 224, 224]), torch.Size([32, 7, 7, 30]))

In [6]:
preds = model(X_batch)
preds.shape

torch.Size([32, 1470])

In [7]:
preds = preds.reshape((preds.shape[0], S, S, B * 5 + C))
preds.shape

torch.Size([32, 7, 7, 30])

In [8]:
decoded_preds = decode_preds(preds)
len(decoded_preds), len(decoded_preds[0])

(32, 98)

In [9]:
CONFIDENCE_THRESHOLD = 0.375
from operator import itemgetter

def filter_sort(decoded_preds):
    sorted_preds = []

    # 1. filter and  by class
    for image in decoded_preds:
        valid_preds = {}
        for pred in image:
            if pred[1] > CONFIDENCE_THRESHOLD:
                class_name = pred[0]
                if class_name in valid_preds:
                    valid_preds[class_name].append(pred)
                else:
                    valid_preds[class_name] = [pred]
    
        sorted_preds.append(valid_preds)

    # 2. sort each class by confidence score 
    for image in sorted_preds:
        for class_name in image:
            image[class_name].sort(key=itemgetter(1), reverse=True)
    
    return sorted_preds

In [10]:
sorted_preds = filter_sort(decoded_preds)
sorted_preds

[{},
 {'horse': [('horse',
    0.4988821600330873,
    tensor(-28.5969, grad_fn=<SubBackward0>),
    tensor(21.6703, grad_fn=<SubBackward0>),
    tensor(7.8209, grad_fn=<AddBackward0>),
    tensor(-24.2459, grad_fn=<AddBackward0>))],
  'chair': [('chair',
    0.45783379539092195,
    tensor(125.8932, grad_fn=<SubBackward0>),
    tensor(101.2084, grad_fn=<SubBackward0>),
    tensor(215.2081, grad_fn=<AddBackward0>),
    tensor(-42.3600, grad_fn=<AddBackward0>))]},
 {'bottle': [('bottle',
    0.5634312544486804,
    tensor(53.1791, grad_fn=<SubBackward0>),
    tensor(-14.7289, grad_fn=<SubBackward0>),
    tensor(70.0413, grad_fn=<AddBackward0>),
    tensor(16.7432, grad_fn=<AddBackward0>))],
  'train': [('train',
    0.47250583305302385,
    tensor(43.2886, grad_fn=<SubBackward0>),
    tensor(-3.9875, grad_fn=<SubBackward0>),
    tensor(81.3909, grad_fn=<AddBackward0>),
    tensor(58.0083, grad_fn=<AddBackward0>)),
   ('train',
    0.40435563019413934,
    tensor(133.6972, grad_fn=<SubBa

In [ ]:
from src.postprocessing import NMS

preds_batch = model(X_batch)
final_preds = NMS(preds_batch)

In [ ]:
final_preds